In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [51]:
import sqlite3

conn = sqlite3.connect('../db.sqlite3')
cursor = conn.cursor()

query = """
SELECT
    oi.id AS orderitem_id,
    o.id AS order_id,
    o.datetime AS order_datetime,
    p.id AS product_id,
    p.name AS product_name,
    p.price AS product_price,
    oi.quantity AS quantity
FROM restaurant_orderitem AS oi
JOIN restaurant_order AS o ON oi.order_id = o.id
JOIN restaurant_product AS p ON oi.product_id = p.id
;
"""

cursor.execute(query)
data = cursor.fetchall()
columns = [desc[0] for desc in cursor.description]

df = pd.DataFrame(data, columns=columns)

df.to_csv('joined_orders.csv', index=False)

print(f"Number of rows: {len(df)}")

conn.close()

df_check = pd.read_csv('joined_orders.csv')
print(df_check.head())
print(f"Rows in CSV: {len(df_check)}")

Number of rows: 74818
   orderitem_id  order_id       order_datetime  product_id  \
0             1         1  2019-08-03 20:25:00           6   
1             2         1  2019-08-03 20:25:00         240   
2             3         1  2019-08-03 20:25:00          19   
3             4         1  2019-08-03 20:25:00          36   
4             5         1  2019-08-03 20:25:00         111   

          product_name  product_price  quantity  
0        Plain Papadum           0.80         2  
1     King Prawn Balti          12.95         1  
2          Garlic Naan           2.95         1  
3        Mushroom Rice           3.95         1  
4  Paneer Tikka Masala           8.95         1  
Rows in CSV: 74818


# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
df_total = df_check.groupby("product_name")["quantity"].sum().sort_values(ascending=False)

df_top10 = df_total.head(10)

plt.figure(figsize=(12, 8))
df_top10.plot.pie(
    autopct="%1.1f%%",
    startangle=90,
    legend=False,
    labels=df_top10.index
)
plt.ylabel("")
plt.title("Top 10 найпопулярніших продуктів за кількістю проданих одиниць")
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df_check["revenue"] = df_check["product_price"] * df_check["quantity"]

popular_products = (
    df_check.groupby("product_name")["revenue"]
    .sum()
    .sort_values(ascending=False)
)
df_top10 = popular_products.head(10)

plt.figure(figsize=(12, 8))
df_top10.plot.pie(
    autopct="%1.1f%%",
    startangle=90,
    legend=False,
    labels=df_top10.index
)
plt.ylabel("")
plt.title("Top 10 pie chart")
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df_check["revenue"] = df_check["product_price"] * df_check["quantity"]
df_check["Data_order"] = pd.to_datetime(df_check['order_datetime'], format='mixed')
df_check["order_hour"] = df_check["Data_order"].dt.hour
popular_hours = (
    df_check.groupby("order_hour")["revenue"]
    .sum()
)


plt.bar(popular_hours.index, popular_hours.values)

plt.title("Restaurant revenue by hour")
plt.xlabel("Hour")
plt.ylabel("Revenue")

plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
df_check["Data_order"] = pd.to_datetime(df_check['order_datetime'], format='mixed')
df_check["order_day"] = df_check["Data_order"].dt.day_name()
popular_days = (
    df_check.groupby("order_day")["revenue"]
    .sum()
)

days_order = [
    "Monday","Tuesday","Wednesday",
    "Thursday","Friday","Saturday","Sunday"
]

popular_days = popular_days.reindex(days_order)
plt.bar(popular_days.index, popular_days.values)

plt.title("Restaurant revenue by day")
plt.xlabel("Day")
plt.ylabel("Revenue")

plt.show()